In [1]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'rides'



In [2]:
from models import Ride, ride_deserializer

In [3]:
import json

In [ ]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-database',
    value_deserializer=ride_deserializer,
    consumer_timeout_ms=5000,  # stop loop after 5s of no new messages
)

In [12]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=15432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = False
cur = conn.cursor()

In [13]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

batch_size = 500
count = 0

try:
    for message in consumer:
        ride = message.value
        pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
        cur.execute(
            """INSERT INTO processed_events
               (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
               VALUES (%s, %s, %s, %s, %s)""",
            (ride.PULocationID, ride.DOLocationID,
             ride.trip_distance, ride.total_amount, pickup_dt)
        )
        count += 1

        if count % batch_size == 0:
            conn.commit()
            print(f"Inserted {count} rows...")

    # Commit any remaining rows when loop ends on timeout.
    conn.commit()
    print(f"Done. Inserted total {count} rows.")

finally:
    consumer.close()
    cur.close()
    conn.close()

Listening to rides and writing to PostgreSQL...
Inserted 500 rows...
Inserted 1000 rows...
Done. Inserted total 1000 rows.
